# GraphSAGE 32-dim Feature Embeddings — Dataset 1 (p-threshold targets)

Reuses the best `graphsage_v1_32` feature embeddings and merges each p-threshold target.

In [1]:
import sys
import os
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.models.embeddings import GNNConfig, extract_embeddings

PROJECT_ROOT = find_project_root()
EMB_DIR      = PROJECT_ROOT / 'src' / 'data' / 'embeddings'
TARGETS_DIR  = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_1' / 'targets'
DATASET2_PATH = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_2'

EMB_DIR.mkdir(parents=True, exist_ok=True)

P_THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

EMB_ROOT = PROJECT_ROOT / 'src' / 'data' / 'embeddings'
def emb_path(name):
    name = str(name)
    ds  = 'dataset_2' if 'dataset2' in name else 'dataset_1'
    sub = 'network_based' if name.startswith('node2vec') else 'feature_based'
    return EMB_ROOT / ds / sub / name


## Config

In [2]:
# Same config as g1_ref graphsage_v1_32
cfg = GNNConfig(hidden_dims=(256, 32), dropout=0.3, lr=0.01, epochs=100, aggregation='mean', device='cpu')

## Dataset 1 — p-threshold targets

Embeddings are already computed in `graphsage_v1_32_srisk_dataset.parquet`.  
For each p threshold, drop the baseline target and merge the p-specific one.

In [3]:
baseline = pd.read_parquet(emb_path('graphsage_v1_32_srisk_dataset.parquet'))
emb_cols = [c for c in baseline.columns if c.startswith('emb_')]
base_keys = ['bank_id', 'year', 'quarter', 'period']
embeddings_d1 = baseline[base_keys + emb_cols].copy()

print(f'Baseline embeddings loaded: {embeddings_d1.shape}')

Baseline embeddings loaded: (145536, 36)


In [4]:
for p in P_THRESHOLDS:
    p_label = f'p{int(p * 100)}'
    p_dir = TARGETS_DIR / p_label

    # Load all quarterly targets for this p threshold
    frames = []
    for csv_path in sorted(p_dir.glob('target_*.csv')):
        df = pd.read_csv(csv_path)
        period = csv_path.stem.replace('target_', '')
        df['period'] = period
        frames.append(df)

    if not frames:
        print(f'{p_label}: no target files found, skipping')
        continue

    targets_p = pd.concat(frames, ignore_index=True)
    merged = embeddings_d1.merge(targets_p, on=['bank_id', 'period'], how='inner')

    out_path = emb_path(f'graphsage_v1_32_{p_label}_dataset.parquet')
    merged.to_parquet(out_path, index=False)
    print(f'{p_label}: shape={merged.shape}  -> {out_path.name}')

p5: shape=(145536, 38)  -> graphsage_v1_32_p5_dataset.parquet
p10: shape=(145536, 38)  -> graphsage_v1_32_p10_dataset.parquet
p15: shape=(145536, 38)  -> graphsage_v1_32_p15_dataset.parquet
p20: shape=(145536, 38)  -> graphsage_v1_32_p20_dataset.parquet
p25: shape=(145536, 38)  -> graphsage_v1_32_p25_dataset.parquet
p30: shape=(145536, 38)  -> graphsage_v1_32_p30_dataset.parquet
p35: shape=(145536, 38)  -> graphsage_v1_32_p35_dataset.parquet
p40: shape=(145536, 38)  -> graphsage_v1_32_p40_dataset.parquet


## Output Summary

In [ ]:
files = sorted((EMB_ROOT / 'dataset_1' / 'feature_based').glob('graphsage_v1_32_p*.parquet'))
for f in files:
    df = pd.read_parquet(f)
    print(f'{f.name:55s}  shape={df.shape}')
